## What is Ring Modulation?

Ring modulation is a sound effect that creates metallic, robotic, and bell-like tones, commonly used in sci-fi sound design, synthesizers. We can create this effect by applying **amplitude modulation** to the audio signal coming from our mic.


## Understanding Amplitude Modulation

You may have heard of amplitude modulation from AM radio, where “AM” stands for amplitude modulation. The same principles used in radio communication can be applied in this workshop to create our sound effect. By multiplying the input of our mic with a periodic function over time, we add ring modulation to our output audio.

The difference equation for AM is:

$$y[n] = x[n] \bullet c[n]$$

where: 
- $x[n]$ is the current sound sample  
- $c[n]$ is a periodic function like $\sin[n]$ or $\cos[n]$

## Ring Modulation in SystemVerilog

### 1. Open `ring_modulation.sv` in Quartus


### 2. Create a Look-Up Table for Sine

Digital hardware like FPGAs cannot easily compute trig functions in real time so we need to create a look-up table (LUT) of precomputed values. 

#### 2.1 Create internal signals and registers

To implement a look-up table (LUT) we need: 
- a index variable that tracks our position in the LUT 
- an array to store our LUT values

Copy the following signals to your module: 
```systemverilog
logic [$clog2(LUT_SIZE)-1:0] index;
logic signed [15:0] sine_lut [0:LUT_SIZE-1];
```

#### 2.2 Load in values

In your module, find:

```systemverilog
// ------------------------------------------------------------------
// Sequential logic
// ------------------------------------------------------------------

// initial load of LUT here
```

This is where we will load the sine lookup table.

Add the following inside this block:

```systemverilog
initial begin
  $readmemh("sin_lut.txt", sine_lut);
end
```

We have provided a lookup table of sine values in hexadecimal format in `sin_lut.txt`. This code loads those values into the LUT array at the very start of execution.

### 3. Add multiplication register

Copy the following register into your module:

```systemverilog
logic signed [31:0] mult_result;
````

### 4. Write the Sequential Logic

#### 4.1 Reset the index to 0
In the `always_ff @(posedge clk)` block, find:

```systemverilog
if (reset) begin
    // Reset logic
end
```

Add the following inside this block:
```systemverilog
index <= 0;
data_out <= 0;
```

#### 4.2 Compute $y[n] = x[n] \bullet sin[n]$

In the `always_ff @(posedge clk)` block, find:

```systemverilog
else if (data_valid) begin
    // Ring modulation logic here

end
```

This is where we will be writing all the logic to create our sound effect. 

**Your Turn:**

Write to `mult_result` (which is $y[n]$) using:
- the current input $x[n]$: `data_in`
- the sine value $c[n]$: `sin[index]`

After that, scale the result by dividing by $2^{15}$ using `>>> 15`, and store it in `data_out`.

>**Note: Fractional numbers in digital hardware**
>
>In digital hardware, we do not store decimals directly because everything is represented as bits. These bits naturally represent integers. To represent fractional values, we scale values by a constant factor. In this case, all LUT values are scaled by $2^{15}$ (Q15 format).
>
>Multiplying two Q15 values produces a Q30 result, so the product is shifted right by 15 bits to return to Q15 scale.


##### 4.3 Update the index 

We need to wrap `index` back to the start once it reaches `LUT_SIZE - 1`, since the sine wave is periodic.

Copy the following into your module: 
```systemverilog
if (index == LUT_SIZE - 1) index <= 0;
else index <= index + 1;
```

5. Result

You should now hear a metallic/robotic tone in your voice when you speak into the mic. You may also notice a deep constant tone in the sound.